# Agent 4: Document Summary Agent

**Purpose:** Generates executive summaries, key insights, and actionable recommendations.

**Tech:** Gemini 2.5 Flash via Vertex AI

In [5]:
from google import genai
from google.genai import types
import json

client = genai.Client(
    vertexai=True,
    project="pwc-agentic-demo",
    location="us-central1"
)

print("Vertex AI client ready!")

Vertex AI client ready!


In [6]:
SUMMARY_PROMPTS = {
    "Invoice": """You are a financial analyst. Generate a concise executive summary of this invoice.
Return as JSON with these exact keys:
- title: string
- one_liner: one sentence summary
- key_highlights: list of 3 highlights
- financial_summary: object with total_amount, tax_rate, payment_deadline
- action_items: list of actions
- risks_or_anomalies: list of concerns

Extracted Data:
{data}""",

    "Contract": """You are a legal analyst. Generate a concise executive summary of this contract.
Return as JSON with these exact keys:
- title: string
- one_liner: one sentence summary
- key_highlights: list of 3 highlights
- obligations: object with party_a list and party_b list
- key_dates: list of objects with event and date
- risks: list of risks
- action_items: list of actions

Extracted Data:
{data}""",

    "Report": """You are a business analyst. Generate a concise executive summary of this report.
Return as JSON with these exact keys:
- title: string
- one_liner: one sentence summary
- key_highlights: list of 3 highlights
- performance_snapshot: object with revenue_trend, growth_areas list, concern_areas list
- insights: list of strategic insights
- action_items: list of actions

Extracted Data:
{data}""",

    "Email": """You are a communication analyst. Generate a concise summary of this email.
Return as JSON with these exact keys:
- title: string
- one_liner: one sentence summary
- key_highlights: list of highlights
- sentiment: positive/neutral/negative
- urgency: high/medium/low
- action_items: list of actions

Extracted Data:
{data}"""
}

print(f"Summary prompts loaded for: {list(SUMMARY_PROMPTS.keys())}")

Summary prompts loaded for: ['Invoice', 'Contract', 'Report', 'Email']


In [7]:
def generate_summary(extracted_data: dict, validation_data: dict, doc_type: str) -> dict:
    if doc_type not in SUMMARY_PROMPTS:
        return {"title": "Summary: Unknown", "one_liner": "Cannot summarize", "key_highlights": [], "action_items": []}
    
    data_str = json.dumps({"extracted_fields": extracted_data, "validation": validation_data}, indent=2, default=str)
    prompt = SUMMARY_PROMPTS[doc_type].replace("{data}", data_str)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.3, response_mime_type="application/json")
    )

    return json.loads(response.text)

In [10]:
invoice_data = {
    "invoice_number": "INV-2024-0042",
    "date": "2024-03-15",
    "vendor": {"name": "TechCorp Solutions Pvt. Ltd.", "address": "HITEC City, Hyderabad"},
    "bill_to": {"name": "PricewaterhouseCoopers", "address": "Cyber Towers, Hyderabad"},
    "line_items": [
        {"description": "AI Consulting Services", "quantity": 40, "rate": "$150/hr", "amount": "$6,000.00"},
        {"description": "Cloud Infrastructure Setup", "quantity": 1, "rate": "$3,500.00", "amount": "$3,500.00"}
    ],
    "subtotal": "$9,500.00",
    "tax": "$1,710.00",
    "total": "$11,210.00",
    "payment_terms": "Net 30 days",
    "bank_details": "HDFC Bank, A/C: 50100012345678"
}

validation = {"is_valid": True, "score": 0.95, "issues": [], "warnings": []}

result = generate_summary(invoice_data, validation, "Invoice")
print(json.dumps(result, indent=2))

{
  "title": "Executive Summary: TechCorp Solutions Invoice INV-2024-0042",
  "one_liner": "This invoice from TechCorp Solutions Pvt. Ltd. for $11,210.00 covers AI consulting and cloud infrastructure services provided to PricewaterhouseCoopers, due by April 14, 2024.",
  "key_highlights": [
    "Invoice from TechCorp Solutions Pvt. Ltd. to PricewaterhouseCoopers for IT services.",
    "Services include 40 hours of AI Consulting and one Cloud Infrastructure Setup.",
    "Total amount due is $11,210.00, which includes an 18% tax on the subtotal."
  ],
  "financial_summary": {
    "total_amount": "$11,210.00",
    "tax_rate": "18%",
    "payment_deadline": "2024-04-14"
  },
  "action_items": [
    "Verify the completion and quality of AI Consulting and Cloud Infrastructure Setup services.",
    "Process payment of $11,210.00 to TechCorp Solutions Pvt. Ltd. by April 14, 2024.",
    "Record the invoice and allocate costs to the relevant project or department."
  ],
  "risks_or_anomalies": [

In [11]:
contract_data = {
    "contract_title": "Master Services Agreement",
    "effective_date": "2024-01-01",
    "parties": [
        {"name": "ABC Technology Pvt. Ltd.", "role": "Provider"},
        {"name": "XYZ Consulting", "role": "Client"}
    ],
    "scope": "AI-powered document processing solutions.",
    "term": "12 months",
    "compensation": "$150,000 annually, payable quarterly",
    "key_terms": [
        {"clause": "Confidentiality", "summary": "2 years post-termination"},
        {"clause": "IP Rights", "summary": "Provider retains pre-existing IP"}
    ],
    "termination_clause": "30 days written notice. Immediate for non-payment."
}

validation = {"is_valid": True, "score": 0.92, "issues": [], "warnings": ["IP clause could be more specific"]}

result = generate_summary(contract_data, validation, "Contract")
print(json.dumps(result, indent=2))

{
  "title": "Master Services Agreement",
  "one_liner": "This Master Services Agreement outlines the provision of AI-powered document processing solutions by ABC Technology Pvt. Ltd. to XYZ Consulting for a 12-month term, with an annual compensation of $150,000.",
  "key_highlights": [
    "ABC Technology Pvt. Ltd. will provide AI-powered document processing solutions to XYZ Consulting.",
    "The agreement has a 12-month term, effective January 1, 2024, with an annual compensation of $150,000, payable quarterly.",
    "Confidentiality obligations extend for two years post-termination, and ABC Technology retains pre-existing intellectual property rights."
  ],
  "obligations": {
    "party_a": [
      "Provide AI-powered document processing solutions.",
      "Retain pre-existing intellectual property.",
      "Maintain confidentiality for 2 years post-termination."
    ],
    "party_b": [
      "Pay $150,000 annually in quarterly installments.",
      "Adhere to confidentiality for 2

In [12]:
report_data = {
    "title": "Quarterly Performance Report - Q4 2023",
    "period": "October 1 - December 31, 2023",
    "executive_summary": "Revenue increased by 23% YoY to $4.2M.",
    "key_metrics": [
        {"metric": "Total Revenue", "value": "$4,200,000", "trend": "up"},
        {"metric": "New Clients", "value": "12", "trend": "up"},
        {"metric": "Client Satisfaction", "value": "4.6/5.0", "trend": "stable"}
    ],
    "challenges": ["Cloud costs exceeded budget", "Two senior engineers resigned"],
    "recommendations": ["Negotiate cloud pricing", "Implement retention bonuses"]
}

validation = {"is_valid": True, "score": 0.97, "issues": [], "warnings": []}

result = generate_summary(report_data, validation, "Report")
print(json.dumps(result, indent=2))

{
  "title": "Quarterly Performance Report - Q4 2023 Executive Summary",
  "one_liner": "Q4 2023 saw a strong 23% year-over-year revenue increase to $4.2M, alongside new client acquisition, despite challenges with cloud costs and staff retention.",
  "key_highlights": [
    "Achieved 23% YoY revenue growth, reaching $4.2M.",
    "Acquired 12 new clients during the quarter.",
    "Maintained high client satisfaction at 4.6/5.0."
  ],
  "performance_snapshot": {
    "revenue_trend": "Upward",
    "growth_areas": [
      "Total Revenue",
      "New Clients"
    ],
    "concern_areas": [
      "Cloud costs exceeding budget",
      "Senior engineer resignations"
    ]
  },
  "insights": [
    "Robust revenue growth and new client acquisition demonstrate strong market demand and effective sales strategies.",
    "Uncontrolled cloud expenses and critical talent loss threaten profitability and operational stability, requiring immediate attention.",
    "Sustained high client satisfaction indic

In [13]:
def classify_document(text: str) -> dict:
    prompt = f"""Classify this document into exactly ONE category: Invoice, Contract, Report, Email, Unknown.
Return JSON with keys: type, confidence, reasoning.
Document:
---
{text}
---"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.1, response_mime_type="application/json")
    )
    return json.loads(response.text)


def extract_fields(text: str, doc_type: str) -> dict:
    prompt = f"""Extract all key fields from this {doc_type} document. Return ONLY JSON.
Document:
---
{text}
---"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.1, response_mime_type="application/json")
    )
    return json.loads(response.text)


def validate_document(extracted_data: dict, doc_type: str) -> dict:
    data_str = json.dumps(extracted_data, indent=2, default=str)
    prompt = f"""Validate this {doc_type} data. Check required fields, date validity, amount calculations.
Return JSON with keys: is_valid (bool), score (0.0-1.0), checks (list), issues (list), warnings (list).
Data:
{data_str}"""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.1, response_mime_type="application/json")
    )
    return json.loads(response.text)


def process_document_full(text: str) -> dict:
    print("\n" + "="*50)
    print("4-AGENT DOCUMENT PROCESSING PIPELINE")
    print("="*50)
    
    print("\nAGENT 1: Classification")
    classification = classify_document(text)
    doc_type = classification["type"]
    print(f"  Type: {doc_type} (confidence: {classification['confidence']})")
    
    print(f"\nAGENT 2: Extraction")
    if doc_type != "Unknown":
        extracted = extract_fields(text, doc_type)
        print(f"  Extracted {len(extracted)} fields")
    else:
        extracted = {}
        print("  Skipped")
    
    print(f"\nAGENT 3: Validation")
    if doc_type != "Unknown" and extracted:
        validation = validate_document(extracted, doc_type)
        print(f"  Status: {'PASSED' if validation['is_valid'] else 'FAILED'} (score: {validation['score']})")
    else:
        validation = {"is_valid": False, "score": 0.0}
        print("  Skipped")
    
    print(f"\nAGENT 4: Summarization")
    if doc_type != "Unknown" and extracted:
        summary = generate_summary(extracted, validation, doc_type)
        print(f"  Title: {summary.get('title', 'N/A')}")
        print(f"  {summary.get('one_liner', 'N/A')}")
        if summary.get("action_items"):
            for item in summary["action_items"]:
                print(f"    - {item}")
    else:
        summary = {}
        print("  Skipped")
    
    print("\n" + "="*50)
    print("PIPELINE COMPLETE")
    print("="*50)
    
    return {"classification": classification, "extracted_fields": extracted, "validation": validation, "summary": summary}

In [14]:
sample = """
INVOICE #INV-2024-0042
Date: March 15, 2024

From: TechCorp Solutions Pvt. Ltd.
123 Innovation Hub, HITEC City, Hyderabad

Bill To: PricewaterhouseCoopers
5th Floor, Block B, Cyber Towers, Hyderabad

Description                    Qty    Rate      Amount
AI Consulting Services          40    $150/hr    $6,000.00
Cloud Infrastructure Setup       1    $3,500.00   $3,500.00

                              Subtotal:   $9,500.00
                              Tax (18%):  $1,710.00
                              Total:     $11,210.00

Payment Terms: Net 30 days
Bank: HDFC Bank, A/C: 50100012345678
"""

result = process_document_full(sample)


4-AGENT DOCUMENT PROCESSING PIPELINE

AGENT 1: Classification
  Type: Invoice (confidence: 1.0)

AGENT 2: Extraction
  Extracted 10 fields

AGENT 3: Validation
  Status: FAILED (score: 0.83)

AGENT 4: Summarization
  Title: Executive Summary: Invoice INV-2024-0042
  This invoice from TechCorp Solutions Pvt. Ltd. to PricewaterhouseCoopers for AI consulting and cloud infrastructure services totals $11,210.00, due by April 14, 2024.
    - Verify the services rendered against the invoice details.
    - Process payment of $11,210.00 to HDFC Bank account 50100012345678 by April 14, 2024.
    - Confirm the validity of the 'unit_price' format for future invoices.

PIPELINE COMPLETE


## Summary Agent - Complete

Next: Streamlit UI + Cloud Run deployment